In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch

from tqdm import tqdm
from transformers import AutoTokenizer



# 1 Define project and gaze-model paths


BASE_DIR = Path.cwd()

REPO_DIR = (
    BASE_DIR
    / "gaze-guided-text-generation"
)

CHECKPOINT_PATH = (
    REPO_DIR
    / "models"
    / "trf_gaze_model.pt"
)

INPUT_PATH = (
    BASE_DIR
    / "clear_test_readabert_final_rerun.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "clear_test_readabert_gaze_rerun.csv"
)

print(
    "Working directory:",
    BASE_DIR
)

print(
    "Gaze repository:",
    REPO_DIR
)

print(
    "Repository exists:",
    REPO_DIR.exists()
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

print(
    "Checkpoint exists:",
    CHECKPOINT_PATH.exists()
)

print(
    "Phase 6 input:",
    INPUT_PATH
)

print(
    "Phase 6 input exists:",
    INPUT_PATH.exists()
)


if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Gaze-model repository not found: {REPO_DIR}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Gaze-model checkpoint not found: {CHECKPOINT_PATH}"
    )

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Phase 6 output not found: {INPUT_PATH}"
    )


# 2 Make the gaze-model repository importable


repo_path = str(
    REPO_DIR.resolve()
)

if repo_path not in sys.path:
    sys.path.insert(
        0,
        repo_path
    )

from modeling.gaze_models import (
    CausalTransformerGazeModel
)

print(
    "\nGaze-model class imported successfully."
)



# 3 Select the available computing device


if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )

print(
    "Device:",
    device
)



# 4 Load the GPT-2 tokenizer used by the gaze model


MODEL_NAME = (
    "openai-community/gpt2"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)

tokenizer.pad_token = (
    tokenizer.eos_token
)

print(
    "\nTokenizer:",
    tokenizer.name_or_path
)



# 5 Load the pretrained gaze-guided transformer


model = (
    CausalTransformerGazeModel
    .from_pretrained(
        MODEL_NAME
    )
)

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    state_dict
)

model.to(
    device
)

model.eval()

print(
    "\nGaze-guided model loaded successfully."
)



# 6 Load the Phase 6 CLEAR test predictions


df = pd.read_csv(
    INPUT_PATH
)

required_columns = [
    "Excerpt",
    "Difficulty_D"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

df = df.dropna(
    subset=[
        "Excerpt"
    ]
).copy()

df["Excerpt"] = (
    df["Excerpt"]
    .astype(str)
)

print(
    "\nNumber of CLEAR passages:",
    len(df)
)

print(
    "\nInput sample:"
)

print(
    df[
        [
            "Excerpt",
            "Difficulty_D"
        ]
    ].head()
)



# 7 Define passage-level gaze prediction


def predict_gaze_scores(
    texts,
    model,
    tokenizer,
    device,
    batch_size=16
):
    model.eval()

    all_scores = []

    for start in tqdm(
        range(
            0,
            len(texts),
            batch_size
        ),
        desc="Predicting gaze"
    ):
        batch_texts = texts[
            start:
            start + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        encoded = {
            key: value.to(
                device
            )
            for key, value
            in encoded.items()
        }

        with torch.no_grad():

            gaze_outputs = model(
                **encoded
            )

            # Remove outputs corresponding to padding tokens
            gaze_outputs = (
                gaze_outputs
                * encoded[
                    "attention_mask"
                ]
            )

            # Sum token-level predictions for each passage
            gaze_scores = (
                gaze_outputs
                .sum(
                    dim=1
                )
            )

        all_scores.extend(
            gaze_scores
            .detach()
            .cpu()
            .tolist()
        )

    return all_scores



# 8 Generate raw gaze scores for all CLEAR passages


texts = (
    df["Excerpt"]
    .tolist()
)

gaze_scores = predict_gaze_scores(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=16
)

if len(
    gaze_scores
) != len(
    df
):
    raise ValueError(
        "Number of gaze predictions does not "
        "match the number of CLEAR passages."
    )

df[
    "Gaze_Score_Raw"
] = gaze_scores

print(
    "\nNumber of predictions:",
    len(
        gaze_scores
    )
)

print(
    "\nFirst 10 gaze predictions:"
)

print(
    df[
        [
            "Excerpt",
            "Difficulty_D",
            "Gaze_Score_Raw"
        ]
    ].head(10)
)



# 9 Check the raw gaze-score output


print(
    "\nRaw gaze-score distribution:"
)

print(
    df[
        "Gaze_Score_Raw"
    ].describe()
)

missing_gaze = (
    df[
        "Gaze_Score_Raw"
    ]
    .isna()
    .sum()
)

print(
    "\nMissing gaze predictions:",
    missing_gaze
)



# 10 Calculate token counts using the gaze tokenizer


def get_token_count(
    text
):
    encoded = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=True
    )

    return len(
        encoded[
            "input_ids"
        ]
    )


df[
    "token_count_gaze"
] = (
    df[
        "Excerpt"
    ]
    .apply(
        get_token_count
    )
)

print(
    "\nGaze-token count distribution:"
)

print(
    df[
        "token_count_gaze"
    ].describe()
)


# 11 Check the preliminary gaze-score/length relationship


gaze_length_correlation = (
    df[
        [
            "Gaze_Score_Raw",
            "token_count_gaze"
        ]
    ]
    .corr(
        method="spearman"
    )
)

print(
    "\nPreliminary Spearman correlation:"
)

print(
    gaze_length_correlation
)



# 12 Check output validity


if (
    df[
        "Gaze_Score_Raw"
    ]
    .isna()
    .any()
):
    raise ValueError(
        "Missing values were found in Gaze_Score_Raw."
    )

if len(df) != 709:
    print(
        "\nWarning: expected 709 CLEAR test passages, "
        f"but found {len(df)}."
    )

else:
    print(
        "\nPassage-count check passed: 709."
    )



# 13 Export the CLEAR data with raw gaze predictions


df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_PATH
)



# 14 Confirm Phase 7 completion


print(
    "\nPHASE 7 COMPLETE"
)

print(
    "Input:",
    INPUT_PATH
)

print(
    "Output:",
    OUTPUT_PATH
)

print(
    "Number of passages:",
    len(df)
)

print(
    "Missing gaze scores:",
    missing_gaze
)

Working directory: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++
Gaze repository: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/gaze-guided-text-generation
Repository exists: True
Checkpoint: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/gaze-guided-text-generation/models/trf_gaze_model.pt
Checkpoint exists: True
Phase 6 input: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/clear_test_readabert_final_rerun.csv
Phase 6 input exists: True

Gaze-model class imported successfully.
Device: mps

Tokenizer: openai-community/gpt2

Gaze-guided model loaded successfully.

Number of CLEAR passages: 709

Input sample:
                                             Excerpt  Difficulty_D
0  And outside before the palace a great garden w...      0.635710
1  One day he had gone beyond any point which he ...      0.443240
2  So off went Lionel to be made a King. He had n...      0.284857
3  Father h

Predicting gaze: 100%|██████████| 45/45 [00:25<00:00,  1.79it/s]



Number of predictions: 709

First 10 gaze predictions:
                                             Excerpt  Difficulty_D  \
0  And outside before the palace a great garden w...      0.635710   
1  One day he had gone beyond any point which he ...      0.443240   
2  So off went Lionel to be made a King. He had n...      0.284857   
3  Father had been away in the country for three ...      0.434188   
4  It was very dark at first, perhaps because the...      0.482255   
5  This is the story of how the swallow's tail ca...      0.249786   
6  The birthday was to be celebrated by great fes...      0.346031   
7  When in the winter they had had their supper a...      0.603776   
8  Ruth was an intellectual looking girl—so peopl...      0.327559   
9  Bess, Belle and Cora were holding this whisper...      0.426578   

   Gaze_Score_Raw  
0       -6.495265  
1      -26.958176  
2      -35.851227  
3      -20.772287  
4      -21.697758  
5      -28.442036  
6      -28.268734  
7      -32.44